<div style="background-color: #2E5F8A; padding: 15px; border-radius: 5px; margin-bottom: 10px;">
    <h2 style="color: white; margin: 0; font-size: 22px;">05. CWC Damage Data Aggregation</h2>
    <p style="color: #cce0ff; margin: 5px 0 0 0; font-size: 13px;">Extracts state-level flood damage data (1953-2023) from CWC Annexure-III, applies documented missing-value conventions, maps states to the four study regions, and aggregates to region-year totals for the AAL window (2000-2020). Source: cwc_flood_damage_statistics_1953_2023.xlsx.</p>
</div>

In [2]:
import openpyxl
import re
import pandas as pd
import numpy as np

DATA_DIR = '/home/ucfadod@ad.ucl.ac.uk/Documents/Dissertation/data/'
OUTPUT_DIR = '/home/ucfadod@ad.ucl.ac.uk/Documents/Dissertation/outputs/'

SRC = DATA_DIR + 'raw/cwc_flood_damage_statistics_1953_2023.xlsx'

<div style="background-color: #2E5F8A; padding: 15px; border-radius: 5px; margin-bottom: 10px;">
    <h2 style="color: white; margin: 0; font-size: 22px;">Part 1: Region Definitions</h2>
    <p style="color: #cce0ff; margin: 5px 0 0 0; font-size: 13px;">Finalized regional groupings (confirmed with Chris). Maharashtra is assigned whole-state to Western_Ghats since CWC damage data is state-level only, with no district breakdown available to support a coastal/interior split.</p>
</div>

In [3]:
STATE_TO_REGION = {
    'KERALA': 'Western_Ghats', 'KARNATAKA': 'Western_Ghats', 'GOA': 'Western_Ghats', 'MAHARASHTRA': 'Western_Ghats',
    'MADHYA PRADESH': 'Central_India', 'CHATTISGARH': 'Central_India', 'ODISHA': 'Central_India', 'JHARKHAND': 'Central_India',
    'UTTAR PRADESH': 'Indo_Gangetic_Plain', 'BIHAR': 'Indo_Gangetic_Plain', 'WEST BENGAL': 'Indo_Gangetic_Plain',
    'PUNJAB': 'Indo_Gangetic_Plain', 'HARYANA': 'Indo_Gangetic_Plain', 'DELHI': 'Indo_Gangetic_Plain',
    'ASSAM': 'Northeast', 'MEGHALAYA': 'Northeast', 'ARUNACHAL PRADESH': 'Northeast', 'TRIPURA': 'Northeast',
}
REGION_STATES = {}
for state, region in STATE_TO_REGION.items():
    REGION_STATES.setdefault(region, []).append(state.title())
N_STATES = {r: len(s) for r, s in REGION_STATES.items()}
print(REGION_STATES)

{'Western_Ghats': ['Kerala', 'Karnataka', 'Goa', 'Maharashtra'], 'Central_India': ['Madhya Pradesh', 'Chattisgarh', 'Odisha', 'Jharkhand'], 'Indo_Gangetic_Plain': ['Uttar Pradesh', 'Bihar', 'West Bengal', 'Punjab', 'Haryana', 'Delhi'], 'Northeast': ['Assam', 'Meghalaya', 'Arunachal Pradesh', 'Tripura']}


<div style="background-color: #2E5F8A; padding: 15px; border-radius: 5px; margin-bottom: 10px;">
    <h2 style="color: white; margin: 0; font-size: 22px;">Part 2: Extraction Functions</h2>
    <p style="color: #cce0ff; margin: 5px 0 0 0; font-size: 13px;">The PDF conversion preserves per-block column positions from merged cells in the source PDF, but positions differ between state blocks (e.g. Arunachal Pradesh wraps across 5 header rows vs. 4 for most states), so columns are detected dynamically per block. Missing-value conventions, verified against the report's own footnotes (repeated on each state table): "Neg" (Negligible) to 0.0... a real, reported value too small to register; "NR" (Not Reported) to NaN; blank cell (also footnoted as "Data Not Reported") to NaN. "Nil" is also mapped to 0.0, following the pattern of whole rows reading "Nil" alongside years with real reported zeros.</p>
</div>

In [4]:
def clean(s):
    return s.replace('\xa0', ' ').replace('\n', ' ') if isinstance(s, str) else s

def to_value(v):
    """Nil -> 0.0, NR/None/blank -> NaN, numeric -> float."""
    if v is None:
        return float('nan')
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if s == '' or s.upper() == 'NR':
        return float('nan')
    if s.upper() in ('NIL', 'NEG'):
        return 0.0
    try:
        return float(s)
    except ValueError:
        return float('nan')

def find_label_col(ws, hr, needle, window=12):
    """Search rows hr+1..hr+window for a cell whose text contains needle. Return its column."""
    for r in range(hr + 1, hr + window + 1):
        for c in range(1, ws.max_column + 1):
            v = ws.cell(row=r, column=c).value
            if v is not None and isinstance(v, str) and needle.lower() in clean(v).lower():
                return c
    return None

def find_value_subcol(ws, hr, group_col, window=12):
    """After the group label's column, find nearest 'Value' text at or after that column."""
    if group_col is None:
        return None
    best = None
    for r in range(hr + 1, hr + window + 1):
        for c in range(group_col, group_col + 15):
            v = ws.cell(row=r, column=c).value
            if v is not None and isinstance(v, str) and 'value' in clean(v).lower():
                if best is None or c < best:
                    best = c
    return best

def find_data_start_and_year_col(ws, hr, window=15):
    """Find first cell == 1953 within a search window after the header row."""
    for r in range(hr + 1, hr + window + 1):
        for c in range(1, 10):
            if ws.cell(row=r, column=c).value == 1953:
                return r, c
    return None, None

def find_state_name(ws, header_row):
    v = ws.cell(row=header_row, column=1).value
    combined = clean(v).upper() if v else ''
    for state in STATE_TO_REGION:
        if state in combined:
            return state
    return None

<div style="background-color: #2E5F8A; padding: 15px; border-radius: 5px; margin-bottom: 10px;">
    <h2 style="color: white; margin: 0; font-size: 22px;">Part 3: Run Extraction</h2>
    <p style="color: #cce0ff; margin: 5px 0 0 0; font-size: 13px;">Known data-quality notes: Himachal Pradesh's block (9/36) failed to convert cleanly, collapsing into unstructured text; not relevant since HP is not one of the 17 study states. Tripura's header contains a typo in the source PDF ("25/37" instead of "25/36"), handled via a looser regex match.</p>
</div>

In [5]:
wb = openpyxl.load_workbook(SRC, data_only=True)
ws = wb['Table 1']

header_rows = []
for r in range(1, ws.max_row + 1):
    v = ws.cell(row=r, column=1).value
    if v and isinstance(v, str) and 'TABLE' in v.upper() and re.search(r'\d+\s*/\s*3\d', v):
        header_rows.append(r)

records = []
skipped = []

for hr in header_rows:
    state = find_state_name(ws, hr)
    if state is None:
        continue
    region = STATE_TO_REGION[state]

    total_col = find_label_col(ws, hr, 'Total damages')
    crops_group_col = find_label_col(ws, hr, 'Damage to Crops')
    houses_group_col = find_label_col(ws, hr, 'Damage to House')
    putil_col = find_label_col(ws, hr, 'Damage to public utilities')
    crops_val_col = find_value_subcol(ws, hr, crops_group_col)
    houses_val_col = find_value_subcol(ws, hr, houses_group_col)
    data_start, year_col = find_data_start_and_year_col(ws, hr)

    if total_col is None or data_start is None:
        skipped.append((hr, state))
        continue

    for i in range(71):
        r = data_start + i
        yr = ws.cell(row=r, column=year_col).value
        if not isinstance(yr, int):
            continue
        records.append({
            'state': state.title(), 'region': region, 'year': yr,
            'total_damages_cr': to_value(ws.cell(row=r, column=total_col).value),
            'crops_damage_cr': to_value(ws.cell(row=r, column=crops_val_col).value) if crops_val_col else np.nan,
            'houses_damage_cr': to_value(ws.cell(row=r, column=houses_val_col).value) if houses_val_col else np.nan,
            'public_util_damage_cr': to_value(ws.cell(row=r, column=putil_col).value) if putil_col else np.nan,
        })

df = pd.DataFrame(records)
print(f"Extracted {len(df)} records (expect {len(STATE_TO_REGION)*71}). Skipped blocks: {skipped}")
print(f"States found ({df.state.nunique()}): {sorted(df.state.unique())}")

df.to_csv(OUTPUT_DIR + 'cwc_extracted_long.csv', index=False)

Extracted 1278 records (expect 1278). Skipped blocks: []
States found (18): ['Arunachal Pradesh', 'Assam', 'Bihar', 'Chattisgarh', 'Delhi', 'Goa', 'Haryana', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Meghalaya', 'Odisha', 'Punjab', 'Tripura', 'Uttar Pradesh', 'West Bengal']


<div style="background-color: #2E5F8A; padding: 15px; border-radius: 5px; margin-bottom: 10px;">
    <h2 style="color: white; margin: 0; font-size: 22px;">Part 4: Regional Aggregation (AAL Window: 2000-2020)</h2>
    <p style="color: #cce0ff; margin: 5px 0 0 0; font-size: 13px;">Window chosen for two reasons: 2020-2023 excluded due to substantially higher missingness (37% vs ~3% for 2000-2019), with several region-year totals built from only 1-2 of 4-5 states reporting. Aggregation uses min_count=1, so a region-year is NaN only if every state in that region is missing that year, correctly handling the Chhattisgarh/MP and Jharkhand/Bihar historical bundling since the parent state's total already includes the split-off state's territory for those years.</p>
</div>

In [6]:
WINDOW = (2000, 2020)
df_w = df[df.year.between(*WINDOW)]

agg = df_w.groupby(['region', 'year'])['total_damages_cr'].agg(
    regional_total_cr=lambda x: x.sum(min_count=1),
    n_reporting=lambda x: x.notna().sum()
).reset_index()
agg['n_states_in_region'] = agg['region'].map(N_STATES)
agg['pct_states_reporting'] = (agg['n_reporting'] / agg['n_states_in_region'] * 100).round(0)

agg['data_quality_flag'] = ''
agg.loc[agg['pct_states_reporting'] < 100, 'data_quality_flag'] = agg.loc[
    agg['pct_states_reporting'] < 100
].apply(lambda r: f"Only {r['n_reporting']}/{r['n_states_in_region']} states reporting", axis=1)

# Known historical context, appended where the generic flag already fired
def append_note(flag, note):
    return f"{flag}; {note}" if flag else note

ci_mask = (agg.region == 'Central_India') & (agg.year == 2000)
agg.loc[ci_mask, 'data_quality_flag'] = agg.loc[ci_mask, 'data_quality_flag'].apply(
    lambda f: append_note(f, 'Jharkhand data embedded in Bihar (IGP) pre-2001; Central_India total likely understated')
)

igp_mask = (agg.region == 'Indo_Gangetic_Plain') & (agg.year == 2000)
agg.loc[igp_mask, 'data_quality_flag'] = agg.loc[igp_mask, 'data_quality_flag'].apply(
    lambda f: append_note(f, 'Bihar total may include former Jharkhand territory; IGP total likely overstated')
)

REGION_ORDER = ['Western_Ghats', 'Central_India', 'Indo_Gangetic_Plain', 'Northeast']
agg['region'] = pd.Categorical(agg['region'], categories=REGION_ORDER, ordered=True)
agg = agg.sort_values(['region', 'year']).reset_index(drop=True)

agg.to_csv(OUTPUT_DIR + 'cwc_regional_aggregated_FINAL_2000_2020.csv', index=False)

print(f"Final aggregation: {len(agg)} rows, {(agg.data_quality_flag != '').sum()} flagged")
agg.pivot(index='year', columns='region', values='regional_total_cr').round(1)

Final aggregation: 84 rows, 15 flagged


region,Western_Ghats,Central_India,Indo_Gangetic_Plain,Northeast
year,,,,
2000,48.4,0.0,5954.0,255.3
2001,97.5,1674.4,4023.0,42.8
2002,61.6,3.1,1655.4,778.7
2003,114.5,1613.3,1812.5,6377.2
2004,0.0,84.5,2401.0,784.4
2005,1896.5,684.2,1040.2,549.0
2006,3272.6,2161.0,168.7,277.5
2007,5475.5,1297.4,2402.0,338.0
2008,656.8,3215.6,1325.7,547.3


In [7]:
agg[agg['data_quality_flag'] != ''][['region', 'year', 'n_reporting', 'n_states_in_region', 'pct_states_reporting', 'data_quality_flag']]

,region,year,n_reporting,n_states_in_region,pct_states_reporting,data_quality_flag
18,Western_Ghats,2018,3,4,75.0,Only 3/4 states reporting
19,Western_Ghats,2019,1,4,25.0,Only 1/4 states reporting
20,Western_Ghats,2020,2,4,50.0,Only 2/4 states reporting
21,Central_India,2000,2,4,50.0,Only 2/4 states reporting; Jharkhand data embe...
40,Central_India,2019,3,4,75.0,Only 3/4 states reporting
42,Indo_Gangetic_Plain,2000,6,6,100.0,Bihar total may include former Jharkhand terri...
47,Indo_Gangetic_Plain,2005,5,6,83.0,Only 5/6 states reporting
57,Indo_Gangetic_Plain,2015,5,6,83.0,Only 5/6 states reporting
58,Indo_Gangetic_Plain,2016,4,6,67.0,Only 4/6 states reporting
59,Indo_Gangetic_Plain,2017,5,6,83.0,Only 5/6 states reporting


<div style="background-color: #2E5F8A; padding: 15px; border-radius: 5px; margin-bottom: 10px;">
    <h2 style="color: white; margin: 0; font-size: 22px;">Summary</h2>
    <p style="color: #cce0ff; margin: 5px 0 0 0; font-size: 13px;">17 states extracted, 1207 state-year records. Approach A selected over Approach B for ERA5 rainfall extraction based on Spearman correlation against this regional damage data (notebook 04)</p>
</div>